In [1]:
import os
import glob
import math
from pathlib import Path


def merge_markdown_files(
    source_dir: str,
    files_per_batch: int,
    prefix: str,
    output_dir: str = None,
    separator: str = '\n\n---\n\n',
    sort_key: str = 'name',          # 'name' | 'mtime' | 'none'
):
    """
    Merge .md files from source_dir into batched output files.

    Parameters
    ----------
    source_dir     : folder chứa các file .md cần merge
    files_per_batch: số file .md gộp vào 1 file output
    prefix         : tiền tố tên file output, eg 'sn' → sn.01.md
    output_dir     : folder xuất file (mặc định = source_dir)
    separator      : chuỗi ngăn cách giữa các file trong batch
    sort_key       : cách sắp xếp file đầu vào ('name', 'mtime', 'none')
    """
    source_dir = Path(source_dir).resolve()
    output_dir = Path(output_dir).resolve() if output_dir else source_dir
    output_dir.mkdir(parents=True, exist_ok=True)

    # ── Thu thập file .md ──────────────────────────────────────────────────
    md_files = list(source_dir.glob('*.md'))

    if not md_files:
        print(f'⚠  Không tìm thấy file .md nào trong: {source_dir}')
        return []

    # ── Sắp xếp ───────────────────────────────────────────────────────────
    if sort_key == 'name':
        md_files.sort(key=lambda p: p.name.lower())
    elif sort_key == 'mtime':
        md_files.sort(key=lambda p: p.stat().st_mtime)
    # 'none' → giữ nguyên thứ tự glob

    total_files   = len(md_files)
    total_batches = math.ceil(total_files / files_per_batch)

    print(f'📂 Source  : {source_dir}')
    print(f'📤 Output  : {output_dir}')
    print(f'📄 Tổng file: {total_files}  │  {files_per_batch} file/batch  │  {total_batches} batch')
    print('─' * 60)

    output_files = []

    for batch_idx in range(total_batches):
        start = batch_idx * files_per_batch
        end   = min(start + files_per_batch, total_files)
        batch = md_files[start:end]

        out_name = f'{prefix}.{batch_idx + 1:02d}.md'
        out_path = output_dir / out_name

        chunks = []
        errors = []

        for filepath in batch:
            try:
                text = filepath.read_text(encoding='utf-8')
            except UnicodeDecodeError:
                # Fallback: thử latin-1 nếu không phải UTF-8
                try:
                    text = filepath.read_text(encoding='latin-1')
                    errors.append(f'  ⚠  {filepath.name}: đọc bằng latin-1 (không phải UTF-8)')
                except Exception as e:
                    errors.append(f'  ✗  {filepath.name}: không đọc được ({e})')
                    continue
            chunks.append(text.rstrip())

        merged = separator.join(chunks)
        if merged and not merged.endswith('\n'):
            merged += '\n'

        out_path.write_text(merged, encoding='utf-8')
        output_files.append(out_path)

        size_kb = out_path.stat().st_size / 1024
        print(f'✅ {out_name}  ({len(batch)} files, {size_kb:.1f} KB)  [files {start+1}–{end}]')
        for f in batch:
            print(f'   • {f.name}')
        for err in errors:
            print(err)
        print()

    print(f'🎉 Hoàn thành! Đã tạo {total_batches} file output tại: {output_dir}')
    return output_files

In [2]:
# ── CẤU HÌNH ──────────────────────────────────────────────────────────────
SOURCE_DIR      = '../docs/kinhtuongung/pali'     # folder chứa các .md gốc
FILES_PER_BATCH = 15            # số file gộp vào 1 output
PREFIX          = 'sn'          # tiền tố tên file output

# Tuỳ chọn:
OUTPUT_DIR      = 'tuongung'          # None = lưu cùng SOURCE_DIR, hoặc eg './output'
SEPARATOR       = '\n\n---\n\n' # dấu phân cách giữa các file
SORT_KEY        = 'name'        # 'name' | 'mtime' | 'none'
# ──────────────────────────────────────────────────────────────────────────

results = merge_markdown_files(
    source_dir      = SOURCE_DIR,
    files_per_batch = FILES_PER_BATCH,
    prefix          = PREFIX,
    output_dir      = OUTPUT_DIR,
    separator       = SEPARATOR,
    sort_key        = SORT_KEY,
)

📂 Source  : /Users/ng/projects/n5/docs/kinhtuongung/pali
📤 Output  : /Users/ng/projects/n5/.scripts/tuongung
📄 Tổng file: 56  │  15 file/batch  │  4 batch
────────────────────────────────────────────────────────────
✅ sn.01.md  (15 files, 684.4 KB)  [files 1–15]
   • sn-001-devatasamyutta.md
   • sn-002-devaputtasamyutta.md
   • sn-003-kosalasamyutta.md
   • sn-004-marasamyutta.md
   • sn-005-bhikkhunisamyutta.md
   • sn-006-brahmasamyutta.md
   • sn-007-brahmanasamyutta.md
   • sn-008-vangisasamyutta.md
   • sn-009-vanasamyutta.md
   • sn-010-yakkhasamyutta.md
   • sn-011-sakkasamyutta.md
   • sn-012-nidanasamyutta.md
   • sn-013-abhisamayasamyutta.md
   • sn-014-dhatusamyutta.md
   • sn-015-anamataggasamyutta.md

✅ sn.02.md  (15 files, 557.7 KB)  [files 16–30]
   • sn-016-kassapasamyutta.md
   • sn-017-labhasakkarasamyutta.md
   • sn-018-rahulasamyutta.md
   • sn-019-lakkhanasamyutta.md
   • sn-020-opammasamyutta.md
   • sn-021-bhikkhusamyutta.md
   • sn-022-khandhasamyutta.md
   • s

---
## 🔍 Kiểm tra nhanh output

In [ ]:
# In 10 dòng đầu của từng file output để kiểm tra
for path in results:
    lines = path.read_text(encoding='utf-8').splitlines()
    preview = '\n'.join(lines[:10])
    print(f'─── {path.name} ({len(lines)} dòng) ───')
    print(preview)
    print('...' if len(lines) > 10 else '')
    print()